# Projector demo: chat with Nemotron before and after fine-tuning

The instructor runs this notebook. Students do not.

1. **Ask.** The notebook opens a plain chat page for the projector. You type messages and Nemotron answers, as it was downloaded.
2. **Label.** Students label the examples in `scenarios/<company>/examples_to_label.csv` (100 messages with empty answer columns) and send the file back.
3. **Fine-tune.** You load their file and run the fine-tuning cell here in the notebook, so the class sees the code and its progress. About 9 to 13 minutes for 100 examples on a free T4.
4. **Switch and ask again.** One more cell switches the chat to the fine-tuned model. Back in the chat, press *Ask the same questions again*.

*To rehearse without a GPU, on a laptop:* `python -m lab.demo --simulate` (answers are simulated).

## 1. Setup

Runtime → Change runtime type → **T4 GPU**, then run:

In [ ]:
# Downloads the lab materials and installs two libraries (~1 minute)
REPO = "https://github.com/manaspandit27-web/manas.git"
BRANCH = "claude/nemotron-finetune-lab"
import os, sys
if not os.path.exists("/content/lab-repo"):
    !git clone -q --depth 1 -b {BRANCH} {REPO} /content/lab-repo
os.chdir("/content/lab-repo/nemotron-lab")
!pip install -q bitsandbytes peft
sys.path.insert(0, ".")
from lab import core, demo
import pandas as pd
print("Ready.")

In [ ]:
# The company for this session. Students label scenarios/<SCENARIO>/examples_to_label.csv
SCENARIO = "airline_complaints"   # others: claims_triage, expense_audit, lead_qualification, invoice_intake

In [ ]:
# Loads NVIDIA Nemotron onto the GPU (~3 minutes)
model, tok = core.load_model()

## 2. Start the chat

The next cell prints the chat's address. Open it in a new tab and put that tab on the projector. The address only works in your own browser, while this notebook is running.

Ask the model a few messages before going on.

In [ ]:
chat = demo.launch(model, tok, scenario=SCENARIO)

## 3. Fine-tune on the class's labels

Run these cells in front of the class, once the students have sent back the examples file.

In [ ]:
# The examples the class labelled: choose the CSV file (or files) the students sent back
from google.colab import files
labelled = chat.read_labelled(files.upload())
# labelled = chat.lab_examples(100)   # no class file? This uses the lab's own labelled examples instead
pd.DataFrame(labelled).head(10)

In [ ]:
# Fine-tune Nemotron on those examples.
# LoRA training: about 1% of the model's weights are trained, the rest stay as downloaded.
losses = chat.finetune(labelled, epochs=2)
core.plot_loss(losses)

## 4. Switch the chat to the fine-tuned model

After this cell, go back to the chat and press *Ask the same questions again*.

In [ ]:
chat.serve("fine-tuned")   # chat.serve("original") switches back